# Drag-and-drop image upload

Run the first four cells in order, or use **Runtime → Run all**. No GPU, API key, or other project files are needed. Upload a picture to see its preview. This notebook implements image upload; story generation can be connected in `preview_upload`.

Gradio opens an embedded interface and a temporary public link while the runtime is active. To stop sharing, run the optional closing cell at the bottom.

In [ ]:
%pip -q install "gradio>=5,<7" "Pillow>=11,<13"

In [ ]:
"""Image decoding shared by the upload interfaces."""

import warnings
from io import BytesIO

from PIL import Image, ImageOps, UnidentifiedImageError

MAX_BYTES = 10 * 1024 * 1024


def load_image(data: bytes) -> Image.Image:
    """Validate one upload and return an oriented RGB image for a pipeline."""
    if len(data) > MAX_BYTES:
        raise ValueError("That picture is too large. Please choose one under 10 MB.")
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", Image.DecompressionBombWarning)
            with Image.open(BytesIO(data)) as source:
                if source.format not in {"JPEG", "PNG", "WEBP"}:
                    raise ValueError("Please choose a JPG, PNG, or WebP picture.")
                source.load()
                return ImageOps.exif_transpose(source).convert("RGB")
    except (UnidentifiedImageError, OSError, Image.DecompressionBombError,
            Image.DecompressionBombWarning) as exc:
        raise ValueError("We couldn't open that picture. Please try another image.") from exc


In [ ]:
from pathlib import Path
import gradio as gr


def preview_upload(filepath):
    if filepath is None:
        return None, "Drop a picture above to get started."
    try:
        with Path(filepath).open("rb") as uploaded:
            image = load_image(uploaded.read(MAX_BYTES + 1))
    except ValueError as exc:
        return None, str(exc)
    # Pass this Pillow RGB `image` to your image-to-text pipeline here.
    return image, "Your picture is ready!"


# Close an earlier instance when rerunning this cell.
if "demo" in globals():
    demo.close()

with gr.Blocks(title="My Story Picture") as demo:
    gr.Markdown("# 📚 My Story Picture\nEvery story starts with a picture. Choose yours!")
    upload = gr.File(
        label="Drag your picture here, or click to choose one",
        file_types=[".jpg", ".jpeg", ".png", ".webp"],
        file_count="single",
        type="filepath",
    )
    gr.Markdown("Choose one JPG, PNG, or WebP picture, up to 10 MB.")
    preview = gr.Image(label="Your story picture", type="pil", interactive=False)
    status = gr.Textbox(value="Drop a picture above to get started.", label="Status", interactive=False)
    upload.change(preview_upload, inputs=upload, outputs=[preview, status])
    clear = gr.Button("Choose another picture")
    clear.click(lambda: (None, None, "Drop a picture above to get started."),
                outputs=[upload, preview, status])

demo.launch(share=True, inline=True, max_file_size="10mb")


## Optional: close the upload app

Change `STOP_APP` to `True` and run this cell when finished.

In [ ]:
STOP_APP = False
if STOP_APP:
    demo.close()
